In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [3]:
data = load_breast_cancer()
df = pd.DataFrame(data.data,columns = data.feature_names)
df["target"] = data.target
print(df.head())
print("shape : ",df.shape)

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  worst area  \
0             

In [4]:
X = df.drop("target",axis=1)
y = df["target"]

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [6]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled,y_train)
knn_pred = knn.predict(X_test_scaled)
print("Actual : ",y_test.values)
print("Predictions : ",knn_pred)

Actual :  [0 1 0 1 0 1 1 0 0 0 1 0 1 0 0 1 1 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 0 0 1
 1 1 0 1 1 1 0 0 1 1 1 1 0 1 1 1 0 1 1 1 0 0 1 1 1 1 1 0 0 1 1 1 1 1 1 1 0
 0 0 0 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 0 0 0 1 0 1 0 1 0 0 0 1 0 0 1 0 1 0 1
 0 1 1]
Predictions :  [0 1 0 0 0 1 1 0 0 0 1 0 1 0 0 1 1 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 0 1 1
 1 0 0 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 0 1 1 1 1 1 1 1 0
 0 0 0 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 0 0 0 1 0 1 0 1 0 0 0 1 0 0 1 0 1 0 1
 0 1 1]


In [7]:
print("Accuracy : ",accuracy_score(y_test,knn_pred))
print("precision : ",precision_score(y_test,knn_pred))
print("recall : ",recall_score(y_test,knn_pred))
print("f1-score : ",f1_score(y_test,knn_pred))

Accuracy :  0.956140350877193
precision :  0.958904109589041
recall :  0.9722222222222222
f1-score :  0.9655172413793104


In [8]:
for k in [1, 3, 5, 7, 9, 11, 15, 20]:

    knn_model = KNeighborsClassifier(
        n_neighbors=k
    )

    knn_model.fit(X_train_scaled, y_train)

    pred = knn_model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, pred)

    print(f"K={k}, Accuracy={accuracy:.4f}")

K=1, Accuracy=0.9386
K=3, Accuracy=0.9825
K=5, Accuracy=0.9561
K=7, Accuracy=0.9737
K=9, Accuracy=0.9737
K=11, Accuracy=0.9737
K=15, Accuracy=0.9737
K=20, Accuracy=0.9737


In [9]:
from sklearn.model_selection import cross_val_score
k_values = [1,3,5,7,9,11,15,20]
for k in k_values:
    knn_m = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_m,X_train_scaled,y_train,cv=5,scoring="accuracy")
    print(f"k = {k}," 
          f"CV Mean Accuracy = {scores.mean():.4f}")

k = 1,CV Mean Accuracy = 0.9451
k = 3,CV Mean Accuracy = 0.9692
k = 5,CV Mean Accuracy = 0.9670
k = 7,CV Mean Accuracy = 0.9692
k = 9,CV Mean Accuracy = 0.9692
k = 11,CV Mean Accuracy = 0.9626
k = 15,CV Mean Accuracy = 0.9604
k = 20,CV Mean Accuracy = 0.9604


In [10]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([("Scaler",StandardScaler()),("knn",KNeighborsClassifier())])

In [11]:
from sklearn.model_selection import GridSearchCV
param_grid = {"knn__n_neighbors":[1,3,5,7,9,11,15,20]}
grid = GridSearchCV(pipeline,param_grid,cv=5,scoring="accuracy",n_jobs=-1)
grid.fit(X_train,y_train)
print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best parameters: {'knn__n_neighbors': 7}
Best CV score: 0.9714285714285715


In [12]:
best_knn = grid.best_estimator_
knn_final_pred = best_knn.predict(X_test)
print("Final KNN Results")

print("Accuracy :", accuracy_score(y_test, knn_final_pred))
print("Precision:", precision_score(y_test, knn_final_pred))
print("Recall   :", recall_score(y_test, knn_final_pred))
print("F1 Score :", f1_score(y_test, knn_final_pred))

Final KNN Results
Accuracy : 0.9736842105263158
Precision: 0.96
Recall   : 1.0
F1 Score : 0.9795918367346939
